# Module 7: Feature Engineering with SQL

**Snowpark ML Fundamentals - Week 2 | Lunch & Learn**

---

## Learning Objectives
- Build time-windowed aggregations (30d/90d/365d)
- Apply deduplication patterns using QUALIFY ROW_NUMBER()
- Compute safe ratios and feature buckets
- Create RFM and behavioral features from raw data

## Key Concept
> **Feature engineering** transforms raw data into ML-ready signals.
> The patterns here (time windows, dedup, ratios, buckets) are the same
> ones used in production dbt feature stores.

---

In [8]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


## 7.1 Setup

In [9]:
import sys
sys.path.insert(0, '../src')

import warnings
warnings.filterwarnings("ignore", category=UserWarning)

from snowpark_fundamentals.session import create_session

session = create_session(env_path='../.env')
print(f"Connected: {session.get_current_database()}.{session.get_current_schema()}")

Connected: "MLDS_Q"."PREDICTIONS"


In [10]:
from snowpark_fundamentals.feature_store.feature_data import (
    create_customer_transactions_dataset,
    create_customer_interactions_dataset,
)

# Ensure source data exists (idempotent)
transactions_df = create_customer_transactions_dataset(session)
interactions_df = create_customer_interactions_dataset(session)
print(f"Transactions: {transactions_df.count()} rows")
print(f"Interactions: {interactions_df.count()} rows")

Transactions: 50000 rows
Interactions: 100000 rows


## 7.2 Deduplication with QUALIFY ROW_NUMBER()

In production, source tables often contain duplicates. The `QUALIFY` clause
is Snowflake's elegant solution — no subqueries needed.

```sql
-- Pattern from NCL stg_customer_360_profile.sql
-- Keeps only the most recent transaction per customer
SELECT * FROM source
QUALIFY ROW_NUMBER() OVER (
    PARTITION BY customer_id
    ORDER BY order_date DESC
) = 1
```

In [11]:
# Deduplication example: keep the most recent transaction per customer
# Without GROUP BY — QUALIFY acts directly on the raw rows
dedup_sql = """
SELECT
    CUSTOMER_ID,
    TRANSACTION_ID,
    ORDER_DATE,
    ORDER_AMOUNT,
    CATEGORY,
    ORDER_STATUS
FROM CUSTOMER_TRANSACTIONS
WHERE ORDER_STATUS = 'COMPLETED'
QUALIFY ROW_NUMBER() OVER (
    PARTITION BY CUSTOMER_ID
    ORDER BY ORDER_DATE DESC
) = 1
"""

deduped = session.sql(dedup_sql)
print(f"Unique customers (latest transaction each): {deduped.count()}")
deduped.show(5)

Unique customers (latest transaction each): 2000
--------------------------------------------------------------------------------------------------
|"CUSTOMER_ID"  |"TRANSACTION_ID"  |"ORDER_DATE"  |"ORDER_AMOUNT"  |"CATEGORY"  |"ORDER_STATUS"  |
--------------------------------------------------------------------------------------------------
|1814           |9752              |2026-02-25    |2733.0          |SPORTS      |COMPLETED       |
|1764           |32321             |2025-12-30    |1329.0          |CLOTHING    |COMPLETED       |
|1656           |33570             |2026-02-16    |3926.0          |TRAVEL      |COMPLETED       |
|388            |36601             |2026-02-07    |1683.0          |SPORTS      |COMPLETED       |
|1942           |23749             |2026-02-12    |4901.0          |SPORTS      |COMPLETED       |
--------------------------------------------------------------------------------------------------



## 7.3 Time-Windowed Aggregations

The core pattern for feature engineering: aggregate at multiple time windows.

```
       30d        90d             365d
  |<--------->|<--------->|<-------- ... -------->|
  now
```

In [12]:
from snowpark_fundamentals.feature_store.feature_data import create_rfm_features

# Create RFM features with 30d/90d/365d time windows
rfm_df = create_rfm_features(session)
print(f"RFM features: {rfm_df.count()} customers")
rfm_df.show(5)

RFM features: 2000 customers
--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
|"CUSTOMER_ID"  |"DAYS_SINCE_LAST_ORDER"  |"ORDERS_30D"  |"ORDERS_90D"  |"ORDERS_365D"  |"ORDERS_TOTAL"  |"SPEND_30D"  |"SPEND_90D"  |"SPEND_365D"  |"SPEND_TOTAL"  |"AVG_ORDER_VALUE"  |"TOTAL_ITEMS"  |"AVG_ITEMS_PER_ORDER"  |"_FEATURE_TIMESTAMP"              |
--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
|818            |47                       |0             |1             |9              |22              |0.0          |2230.0       |22114.0       |52443.0        |2383.77            |119

In [13]:
# Inspect the schema — all numeric, ready for ML
for field in rfm_df.schema.fields:
    print(f"  {field.name}: {field.datatype}")

  CUSTOMER_ID: LongType()
  DAYS_SINCE_LAST_ORDER: LongType()
  ORDERS_30D: LongType()
  ORDERS_90D: LongType()
  ORDERS_365D: LongType()
  ORDERS_TOTAL: LongType()
  SPEND_30D: DoubleType()
  SPEND_90D: DoubleType()
  SPEND_365D: DoubleType()
  SPEND_TOTAL: DoubleType()
  AVG_ORDER_VALUE: DoubleType()
  TOTAL_ITEMS: LongType()
  AVG_ITEMS_PER_ORDER: DecimalType(21, 2)
  _FEATURE_TIMESTAMP: TimestampType(timezone=TimestampTimeZone('ltz'))


## 7.4 Behavioral Feature Aggregations

In [14]:
from snowpark_fundamentals.feature_store.feature_data import create_behavioral_features

# Create behavioral features from interactions
behavioral_df = create_behavioral_features(session)
print(f"Behavioral features: {behavioral_df.count()} customers")
behavioral_df.show(5)

Behavioral features: 2000 customers
-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
|"CUSTOMER_ID"  |"TOTAL_PAGE_VIEWS"  |"TOTAL_CLICKS"  |"TOTAL_SUPPORT_TICKETS"  |"TOTAL_EMAIL_ENGAGEMENTS"  |"INTERACTIONS_30D"  |"SUPPORT_TICKETS_30D"  |"INTERACTIONS_90D"  |"DAYS_SINCE_LAST_INTERACTION"  |"PREFERRED_CHANNEL"  |"AVG_DURATION_SECONDS"  |"_FEATURE_TIMESTAMP"              |
-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
|928            |15                  |10              |9                        |23           

## 7.5 Derived Features: Ratios & Buckets

Derived features combine base features using safe division and bucketing.

**Safe ratio pattern** (from NCL l2_contact_derived_features.sql):
```sql
CASE
    WHEN denominator > 0
    THEN ROUND(numerator::FLOAT / denominator, 4)
    ELSE 0
END
```

In [15]:
from snowpark_fundamentals.feature_store.feature_data import create_derived_features

# Create derived features combining RFM + behavioral
derived_df = create_derived_features(session)
print(f"Derived features: {derived_df.count()} customers")
derived_df.show(5)

Derived features: 2000 customers
--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
|"CUSTOMER_ID"  |"DAYS_SINCE_LAST_ORDER"  |"ORDERS_30D"  |"ORDERS_90D"  |"ORDERS_365D"  |"ORDERS_TOTAL"  |"SPEND_TOTAL"  |"AVG_ORDER_VALUE"  |"TOTAL_PAGE_VIEWS"  |"TOTAL_CLICKS"  |"TOTAL_SUPPORT_TICKETS"  |"INTERACTIONS_30D"  |"DAYS_SINCE_LAST_INTERACTION"  |"PREFERRED_CHANNEL"  |"SPEND_PER_ORDER"  |"ORDER_RECENCY_RATIO"  |"CLICK_THROUGH_RATE"  |"RECENCY_BUCKET"  |"SPEND_BUCKET"  |"ENGAGEMENT_SCORE"  |"_FEATURE_TIMESTAMP"              |
-------------------------------------------------------------------------------------

In [16]:
from snowflake.snowpark import functions as F

# Distribution of recency buckets
derived_df.group_by("RECENCY_BUCKET").agg(
    F.count("*").alias("COUNT")
).sort("COUNT", ascending=False).show()

------------------------------
|"RECENCY_BUCKET"  |"COUNT"  |
------------------------------
|ACTIVE            |1048     |
|WARM              |749      |
|COOLING           |186      |
|AT_RISK           |17       |
------------------------------



In [17]:
# Distribution of spend buckets
derived_df.group_by("SPEND_BUCKET").agg(
    F.count("*").alias("COUNT")
).sort("COUNT", ascending=False).show()

----------------------------
|"SPEND_BUCKET"  |"COUNT"  |
----------------------------
|HIGH            |2000     |
----------------------------



## Key Takeaways

1. **Deduplication** with `QUALIFY ROW_NUMBER()` is cleaner than subqueries
2. **Time-windowed aggregations** (30d/90d/365d) capture recency and trends
3. **Safe ratios** prevent divide-by-zero errors in derived features
4. **Bucketing** converts continuous features to categorical bins for interpretability
5. These exact patterns are used in **production dbt feature stores**

---

**Next: [08 - Feature Views](08_feature_views.ipynb)**

In [18]:
session.close()